In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "4" #  set the number of threads for OpenMP and MKL to 4
os.environ["MKL_NUM_THREADS"] = "4" # set the number of threads for OpenMP and MKL to 4

import torch

torch.set_num_threads(4)

In [ ]:
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ultralytics
import yaml
from ultralytics import YOLO

In [ ]:
ultralytics.checks()

In [ ]:
cwd = Path.cwd()
p = cwd
while p != p.parent and p.name != "lesson_21":
    p = p.parent

ROOT = p if p.name == "lesson_21" else cwd  # fallback
print("CWD:", cwd)
print("ROOT:", ROOT.resolve())

In [ ]:
# Make paths robust: notebook is in course_work/training => parent is course_work
# ROOT = Path.cwd().parent

DATASET_SRC = ROOT / "data" / "coin_dataset"
IMAGES_SRC = DATASET_SRC / "images"
LABELS_SRC = DATASET_SRC / "labels"

print("CWD:", Path.cwd())
print("ROOT:", ROOT.resolve())
print("IMAGES_SRC:", IMAGES_SRC.resolve())
print("LABELS_SRC:", LABELS_SRC.resolve())

BASE_MODEL = 'yolo26s.pt'

In [ ]:
assert IMAGES_SRC.exists(), IMAGES_SRC
assert LABELS_SRC.exists(), LABELS_SRC

names = ["1 cent","2 cent","5 cent","10 cent","20 cent","50 cent","1 euro","2 euro"]
nc = len(names)

In [ ]:
DATASET_SRC

In [ ]:
OUT = ROOT / "data" / "coin_yolo_split"
images_out = OUT / "images"
labels_out = OUT / "labels"

for split in ["train", "val", "test"]:
    (images_out / split).mkdir(parents=True, exist_ok=True)
    (labels_out / split).mkdir(parents=True, exist_ok=True)

# Build pairs
image_files = sorted(IMAGES_SRC.glob("*.jpg"))
pairs = []
for img in image_files:
    lab = LABELS_SRC / f"{img.stem}.txt"
    if lab.exists():
        pairs.append((img, lab))

print("Found pairs:", len(pairs))
assert len(pairs) > 0

# Skip copying if already done
already = list((images_out / "train").glob("*.jpg"))
if len(already) == 0:
    random.seed(42)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)

    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]

    def copy_pairs(pairs_list, split):
        for img, lab in pairs_list:
            shutil.copy2(img, images_out / split / img.name)
            shutil.copy2(lab, labels_out / split / lab.name)

    copy_pairs(train_pairs, "train")
    copy_pairs(val_pairs, "val")
    copy_pairs(test_pairs, "test")
    print("Copied train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))
else:
    print("Split already exists. train images:", len(already))



# Create yaml file

In [ ]:
data_yaml = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": nc,
    "names": names,
}

yaml_path = OUT / "data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("Wrote:", yaml_path)

# Train

- https://docs.ultralytics.com/modes/train/

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
model = YOLO(BASE_MODEL)

results = model.train(
    data=str(yaml_path),

    # Core training
    epochs=25,              # ↓ from 50 (CPU realistic)
    imgsz=256,              # 🔑 critical balance
    batch=16,

    # Device
    device=device,

    # Stability
    workers=4,
    cache=False,
    amp=True,

    # Augmentations (controlled)
    mosaic=0.0,             # keep OFF (huge RAM spikes)
    mixup=0.0,

    # Light augmentations only
    scale=0.3,
    translate=0.05,
    shear=1.0,
    perspective=0.0,

    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,

    # Logging
    verbose=True
)

#  Statistics

In [ ]:
run_dir = Path(results.save_dir)

best_pt = run_dir / "weights" / "best.pt"

if best_pt.exists():
    print("Best weights:", best_pt)
else:
    print("Best weights not found. Available files:", list((run_dir / "weights").glob("*")))

In [ ]:
eval_model = YOLO(str(best_pt))
val_metrics = eval_model.val(data=str(yaml_path),
                             split="val",
                             imgsz=512,
                             device=device)
print(val_metrics)

In [ ]:
# Display Ultralytics-saved confusion matrix image (most reliable)
cm_png = run_dir / "confusion_matrix.png"
if cm_png.exists():
    from PIL import Image
    img = Image.open(cm_png)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (Ultralytics)")
    plt.show()
else:
    print("confusion_matrix.png not found in:", run_dir)

In [ ]:
box = val_metrics.box

# Per-class mAP50-95 is commonly available as `box.maps`
maps = getattr(box, "maps", None)

rows = []
if maps is not None:
    for i, c in enumerate(names):
        rows.append({"class": c, "mAP50-95": float(maps[i])})
    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
    display(df)
else:
    print("Could not find per-class `maps` on val_metrics.box. Available:", dir(box))

In [ ]:
for k in ["p", "r", "map50", "map"]:
    v = getattr(box, k, None)
    print(k, v)

In [ ]:
# Evaluate at one image
run_dir = Path(results.save_dir)
best_pt = run_dir / "weights" / "best.pt"

assert best_pt.exists(), best_pt
print("Best weights:", best_pt)

eval_model = YOLO(str(best_pt))

In [ ]:
path = '../../data/coin_dataset/images/018.jpg'

eval_model.predict(source=path,
                   show=True,
                   save=True)

In [ ]:
path = '../../data/test/beach_01.jpg'

eval_model.predict(source=path,
                   show=True,
                   save=True)